# 真实 API：精确匹配一致率

通过真实 OpenAI Responses API 生成原回答及独立样本，计算采样一致率。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai]"`；配置 `OPENAI_API_KEY` 会产生真实 API 调用。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `OPENAI_MODEL: str`：Responses API 模型名。
- `TEMPERATURE: float = 0.8`：采样多样性。
- `NUM_SAMPLES: int = 5`：独立样本数。

## 步骤 1：配置真实 API

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

## 步骤 2：采样一致率

In [ ]:
from hallucination.detection.pipelines.short_answer import BlackBoxSamplingPipeline
NUM_SAMPLES = 5
result = BlackBoxSamplingPipeline(backend, num_samples=NUM_SAMPLES, threshold=0.5).detect_prompt("蓝鲸属于什么动物类群？只回答一个简短分类名称。")
print(result.metadata["response"])
print(result.metadata["samples"])
print("exact-match risk:", result.risk)

## 结果解读与限制

精确匹配仅适合封闭短答案。同义改写会被低估；采样一致率是自一致性信号，不是事实核查。